# 02 Preprocess alphabet
Crop with `HandCropper`, save 224x224 uint8 crops, zip to Drive. Sample up to 1500 images per class. Skip `nothing`.


In [ ]:
# 1) Mount Google Drive (a popup asks permission)
from google.colab import drive
drive.mount("/content/drive")

# 2) Paths
DRIVE = "/content/drive/MyDrive/sign-language"

# 3) Code + packages
!git clone https://github.com/<you>/sign-language-translator.git /content/repo 2>/dev/null || git -C /content/repo pull
!pip install -q mediapipe==0.10.14 "numpy<2" scikit-learn seaborn tqdm
%cd /content/repo
import sys
sys.path.insert(0, "/content/repo")



In [ ]:
!mkdir -p ~/.kaggle
!cp {DRIVE}/kaggle/kaggle.json ~/.kaggle/
!chmod 600 ~/.kaggle/kaggle.json
!pip install -q kaggle
!kaggle datasets download -d grassknoted/asl-alphabet -p /content/raw
!unzip -q /content/raw/asl-alphabet.zip -d /content/raw/asl
!ls /content/raw/asl



In [ ]:
import cv2, random, os
from pathlib import Path
from tqdm import tqdm
from src.preprocess import HandCropper
from src.config import ALPHABET_LABELS

SRC = Path("/content/raw/asl/asl_alphabet_train/asl_alphabet_train")  # adjust if needed
OUT = Path("/content/processed"); OUT.mkdir(exist_ok=True)
PER_CLASS = 1500
cropper = HandCropper(static_image_mode=True)

stats = {}
for label in ALPHABET_LABELS:
    files = sorted((SRC / label).glob("*.jpg"))
    random.seed(0); random.shuffle(files)
    (OUT / label).mkdir(exist_ok=True)
    ok = 0
    for f in tqdm(files[:PER_CLASS], desc=label):
        img = cv2.imread(str(f))
        crop, _ = cropper.crop(img)
        if crop is None: continue
        crop = cv2.resize(crop, (224, 224))
        cv2.imwrite(str(OUT / label / f"{ok}.jpg"), cv2.cvtColor(crop, cv2.COLOR_RGB2BGR))
        ok += 1
    stats[label] = ok
print("detection success counts (not the original file count):", stats)
cropper.close()

!cd /content && zip -qr processed_alphabet.zip processed
!cp /content/processed_alphabet.zip {DRIVE}/processed/

